# 01 · Tabular pipeline (guided)

**Style A: syntax gaps.** A complete, runnable notebook with 8 gaps. Each gap is a `# TODO(gN)` comment
followed by a placeholder containing `____`. An unfilled gap raises `NameError`. The cell after each gap
checks it. Solutions: `../solutions/guided/01_tabular_pipeline.md`.

**Traces to** `notes/ioai-task-patterns.md`: P1 (skeleton), P2 (exact submission format, no header),
T25-ANTIQUE (CSV labels without a header, zipped A/B files), T24-O-BOBAI (macro F1).

Runtime: under 1 minute on CPU.

In [ ]:
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay

SEED = 0

## 1. Load and engineer a few features

`boat`, `body` and `home.dest` are recorded *after* the outcome (lifeboat number, body number), so they leak
the label. Drop them.

In [ ]:
titanic = fetch_openml("titanic", version=1, as_frame=True)
df = titanic.frame.drop(columns=["boat", "body", "home.dest"])
df["survived"] = df["survived"].astype(int)

df["title"] = df["name"].str.extract(r",\s*([^\.]+)\.", expand=False)
df["title"] = df["title"].where(df["title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
df["deck"] = df["cabin"].str[0].fillna("U")
df["family"] = df["sibsp"] + df["parch"] + 1

num_cols = ["age", "fare", "sibsp", "parch", "family"]
cat_cols = ["pclass", "sex", "embarked", "title", "deck"]
X = df[num_cols + cat_cols].copy()
X[cat_cols] = X[cat_cols].astype(object)   # plain objects: SimpleImputer/OneHotEncoder handle NaN cleanly
y = df["survived"]
print(X.shape, y.mean().round(3))
X.head()

## 2. Stratified split

In [ ]:
# TODO(g1): stratified 80/20 split that keeps the survival ratio; random_state=SEED
X_train, X_test, y_train, y_test = train_test_split(X, y, ____)

In [ ]:
assert len(X_test) == 262, len(X_test)
assert abs(y_train.mean() - y_test.mean()) < 0.005, "not stratified"
print("g1 ok")

## 3. Preprocessing inside a `ColumnTransformer`

Everything that learns statistics (medians, means, categories) lives in the pipeline, so it is fitted on the
training fold only.

In [ ]:
numeric = Pipeline([
    # TODO(g2): median imputation, then standardisation (two named steps)
    ____,
])
categorical = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    # TODO(g3): one-hot encoder that does not crash on categories unseen during fit
    ("onehot", ____),
])
# sparse_threshold=0 -> dense output (HistGradientBoosting refuses sparse matrices)
preprocess = ColumnTransformer([("num", numeric, num_cols), ("cat", categorical, cat_cols)], sparse_threshold=0)

In [ ]:
Xt = preprocess.fit_transform(X_train)
assert not np.isnan(Xt).any(), "NaNs left after imputation"
assert np.allclose(Xt[:, :len(num_cols)].mean(0), 0, atol=1e-6), "numeric columns not standardised"
weird = X_test.head(1).copy(); weird["deck"] = "Z"          # never seen in training
preprocess.transform(weird)
print("g2, g3 ok:", Xt.shape)

## 4. Logistic regression baseline, scored with ROC AUC

In [ ]:
clf = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))]).fit(X_train, y_train)
# TODO(g4): ROC AUC needs the probability of the positive class, not hard labels
proba = ____
auc = roc_auc_score(y_test, proba)
print(f"test ROC AUC = {auc:.3f}")

In [ ]:
assert proba.shape == (len(X_test),), proba.shape
assert len(np.unique(proba)) > 50, "looks like hard labels, not probabilities"
assert auc > 0.80
print("g4 ok")

## 5. Cross-validation of the whole pipeline

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# TODO(g5): 5-fold CV of clf on the TRAINING split, scored by ROC AUC
scores = ____
print(scores.round(3), scores.mean().round(3))

In [ ]:
assert scores.shape == (5,) and scores.mean() > 0.80
print("g5 ok")

## 6. Grid search over a gradient-boosting pipeline

In [ ]:
hgb = Pipeline([("prep", preprocess), ("model", HistGradientBoostingClassifier(random_state=SEED))])
# TODO(g6): grid keys for a Pipeline step are "<step name>__<parameter>"
grid = {____: [3, 5, None], ____: [0.05, 0.1]}
search = GridSearchCV(hgb, grid, cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
print(search.best_params_, round(search.best_score_, 3))

In [ ]:
assert set(search.best_params_) == {"model__max_depth", "model__learning_rate"}
assert search.best_score_ > 0.82
print("g6 ok")

## 7. Macro F1 and the confusion matrix

In [ ]:
proba = search.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
# TODO(g7): macro-averaged F1 (mean of per-class F1, each class weighted equally)
f1 = ____
print(f"macro F1 = {f1:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred)).plot(ax=ax[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, proba, ax=ax[1])
plt.tight_layout(); plt.show()

In [ ]:
(tn, fp), (fn, tp) = confusion_matrix(y_test, pred)
f1_pos = 2 * tp / (2 * tp + fp + fn); f1_neg = 2 * tn / (2 * tn + fn + fp)
assert np.isclose(f1, (f1_pos + f1_neg) / 2), "that is not the macro average"
print("g7 ok")

## 8. Submission files in the IOAI format

T25-ANTIQUE wanted `submissionA.csv` with one label per line and **no header** (a header row would be
parsed as a prediction and break the file), zipped together with `submissionB.csv`.

In [ ]:
# TODO(g8): write `pred` as one integer per line, no header, no index
pd.Series(pred).to_csv("submissionA.csv", ____)
pd.Series(pred).to_csv("submissionB.csv", index=False, header=False)   # pretend test B = test A here
with zipfile.ZipFile("submission.zip", "w") as z:
    for f in ["submissionA.csv", "submissionB.csv"]:
        z.write(f)

In [ ]:
lines = open("submissionA.csv").read().splitlines()
assert len(lines) == len(X_test) and set(lines) <= {"0", "1"}, lines[:3]
assert zipfile.ZipFile("submission.zip").namelist() == ["submissionA.csv", "submissionB.csv"]
print("g8 ok, all gaps done")